In [ ]:
from __future__ import annotations
 
# =================================================================================
#  IMPORTS
# =================================================================================
import json
import queue
import threading
import traceback
import tkinter as tk
from tkinter import ttk, messagebox, filedialog
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from pathlib import Path
from typing import Optional, Callable
 
import numpy as np
import pandas as pd
 
import matplotlib
matplotlib.use("TkAgg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.figure import Figure
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg, NavigationToolbar2Tk
from matplotlib.patches import Rectangle
import seaborn as sns
 
try:
    import yfinance as yf
except ImportError:  # pragma: no cover
    yf = None
 
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.formatting.rule import ColorScaleRule
from openpyxl.chart import LineChart, Reference
from openpyxl.utils import get_column_letter
 
 
# =================================================================================
#  CONSTANTS / THEME  (constants.py)
# =================================================================================
class Theme:
    """Central design-token store: colors, fonts, spacing for the dark UI."""
 
    # --- Palette -----------------------------------------------------------
    BG_PRIMARY = "#0f172a"      # app background (slate-900)
    BG_SECONDARY = "#111827"    # panel background
    BG_SIDEBAR = "#161b2e"
    BG_CARD = "#1e293b"         # card / widget background (slate-800)
    BG_CARD_HOVER = "#26334a"
    BG_INPUT = "#0b1220"
 
    ACCENT = "#3b82f6"          # blue-500
    ACCENT_HOVER = "#2563eb"
    ACCENT_GREEN = "#22c55e"
    ACCENT_RED = "#ef4444"
    ACCENT_AMBER = "#f59e0b"
    ACCENT_PURPLE = "#a855f7"
 
    TEXT_PRIMARY = "#e5e7eb"
    TEXT_SECONDARY = "#94a3b8"
    TEXT_MUTED = "#64748b"
    TEXT_ON_ACCENT = "#ffffff"
 
    BORDER = "#273449"
 
    # --- Typography ----------------------------------------------------------
    FONT_FAMILY = "Segoe UI"
    FONT_TITLE = (FONT_FAMILY, 22, "bold")
    FONT_SUBTITLE = (FONT_FAMILY, 12)
    FONT_H1 = (FONT_FAMILY, 18, "bold")
    FONT_H2 = (FONT_FAMILY, 13, "bold")
    FONT_BODY = (FONT_FAMILY, 10)
    FONT_BODY_BOLD = (FONT_FAMILY, 10, "bold")
    FONT_SMALL = (FONT_FAMILY, 9)
    FONT_MONO = ("Consolas", 10)
    FONT_METRIC_VALUE = (FONT_FAMILY, 17, "bold")
    FONT_METRIC_LABEL = (FONT_FAMILY, 9)
 
    # --- Layout ----------------------------------------------------------
    SIDEBAR_WIDTH = 220
    PAD = 14
    RADIUS = 14
 
 
PERIOD_OPTIONS = {
    "1 Month": "1mo",
    "3 Months": "3mo",
    "6 Months": "6mo",
    "1 Year": "1y",
    "5 Years": "5y",
    "Max": "max",
}
 
try:
    _APP_DIR = Path(__file__).resolve().parent
except NameError:
    # __file__ is undefined when running inside a Jupyter/IPython cell.
    _APP_DIR = Path.cwd()
APP_DATA_FILE = _APP_DIR / "app_data.json"
MARKET_INDEX_TICKER = "^GSPC"
 
sns.set_theme(style="darkgrid")
plt.rcParams.update({
    "figure.facecolor": Theme.BG_CARD,
    "axes.facecolor": Theme.BG_CARD,
    "savefig.facecolor": Theme.BG_CARD,
    "axes.edgecolor": Theme.BORDER,
    "axes.labelcolor": Theme.TEXT_PRIMARY,
    "xtick.color": Theme.TEXT_SECONDARY,
    "ytick.color": Theme.TEXT_SECONDARY,
    "text.color": Theme.TEXT_PRIMARY,
    "grid.color": Theme.BORDER,
    "grid.alpha": 0.4,
    "font.family": "sans-serif",
    "axes.titlecolor": Theme.TEXT_PRIMARY,
    "legend.facecolor": Theme.BG_CARD,
    "legend.edgecolor": Theme.BORDER,
})
 
 
# =================================================================================
#  UTILS MODULE  (utils/persistence.py, utils/validation.py, utils/tooltip.py)
# =================================================================================
class AppStorage:
    """Lightweight JSON-backed persistence for watchlist / recent searches / history."""
 
    def __init__(self, path: Path = APP_DATA_FILE):
        self.path = path
        self.data = {
            "watchlist": [],
            "recent_searches": [],
            "analysis_history": [],
        }
        self._load()
 
    def _load(self) -> None:
        if self.path.exists():
            try:
                with open(self.path, "r", encoding="utf-8") as fh:
                    loaded = json.load(fh)
                    self.data.update(loaded)
            except (json.JSONDecodeError, OSError):
                pass
 
    def save(self) -> None:
        try:
            with open(self.path, "w", encoding="utf-8") as fh:
                json.dump(self.data, fh, indent=2)
        except OSError:
            pass
 
    def add_recent_search(self, ticker: str) -> None:
        rs = self.data["recent_searches"]
        ticker = ticker.upper()
        if ticker in rs:
            rs.remove(ticker)
        rs.insert(0, ticker)
        self.data["recent_searches"] = rs[:10]
        self.save()
 
    def toggle_watchlist(self, ticker: str) -> bool:
        """Returns True if now watchlisted, False if removed."""
        wl = self.data["watchlist"]
        ticker = ticker.upper()
        if ticker in wl:
            wl.remove(ticker)
            self.save()
            return False
        wl.append(ticker)
        self.save()
        return True
 
    def is_watchlisted(self, ticker: str) -> bool:
        return ticker.upper() in self.data["watchlist"]
 
    def add_history(self, entry: dict) -> None:
        hist = self.data["analysis_history"]
        hist.insert(0, entry)
        self.data["analysis_history"] = hist[:50]
        self.save()
 
 
def validate_ticker(raw: str) -> Optional[str]:
    """Basic sanity validation for a ticker symbol input."""
    if not raw:
        return None
    ticker = raw.strip().upper()
    if not ticker:
        return None
    allowed = set("ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789.-^")
    if not all(ch in allowed for ch in ticker):
        return None
    if len(ticker) > 12:
        return None
    return ticker
 
 
class Tooltip:
    """Simple hover tooltip for any Tkinter widget."""
 
    def __init__(self, widget: tk.Widget, text: str, delay: int = 450):
        self.widget = widget
        self.text = text
        self.delay = delay
        self.tip_window: Optional[tk.Toplevel] = None
        self._after_id: Optional[str] = None
        widget.bind("<Enter>", self._schedule)
        widget.bind("<Leave>", self._hide)
        widget.bind("<ButtonPress>", self._hide)
 
    def _schedule(self, _event=None) -> None:
        self._after_id = self.widget.after(self.delay, self._show)
 
    def _show(self) -> None:
        if self.tip_window or not self.text:
            return
        x = self.widget.winfo_rootx() + 20
        y = self.widget.winfo_rooty() + self.widget.winfo_height() + 8
        self.tip_window = tw = tk.Toplevel(self.widget)
        tw.wm_overrideredirect(True)
        tw.wm_geometry(f"+{x}+{y}")
        label = tk.Label(
            tw, text=self.text, justify="left", background="#0b1220",
            foreground=Theme.TEXT_PRIMARY, relief="solid", borderwidth=1,
            font=Theme.FONT_SMALL, padx=8, pady=4,
        )
        label.pack()
 
    def _hide(self, _event=None) -> None:
        if self._after_id:
            self.widget.after_cancel(self._after_id)
            self._after_id = None
        if self.tip_window:
            self.tip_window.destroy()
            self.tip_window = None
 
 
def fmt_num(value, decimals: int = 2, suffix: str = "") -> str:
    """Safely format a numeric value, gracefully handling None/NaN."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return "N/A"
    try:
        return f"{value:,.{decimals}f}{suffix}"
    except (ValueError, TypeError):
        return "N/A"
 
 
def fmt_large_number(value) -> str:
    """Format large numbers (market cap, volume) using K/M/B/T suffixes."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return "N/A"
    value = float(value)
    for unit, div in (("T", 1e12), ("B", 1e9), ("M", 1e6), ("K", 1e3)):
        if abs(value) >= div:
            return f"{value / div:,.2f}{unit}"
    return f"{value:,.0f}"
 
 
# =================================================================================
#  DATA MODULE  (data/fetcher.py)
# =================================================================================
@dataclass
class StockDataset:
    """Container bundling everything fetched/derived for one ticker."""
 
    ticker: str
    history: pd.DataFrame
    info: dict = field(default_factory=dict)
    market_history: Optional[pd.DataFrame] = None
 
 
class DataFetcher:
    """Thin wrapper around yfinance with basic caching and error handling."""
 
    def __init__(self):
        self._cache: dict[str, StockDataset] = {}
 
    def fetch(self, ticker: str, period: str, with_market: bool = True) -> StockDataset:
        if yf is None:
            raise RuntimeError(
                "The 'yfinance' package is not installed. Run: pip install yfinance"
            )
 
        cache_key = f"{ticker}_{period}"
        if cache_key in self._cache:
            return self._cache[cache_key]
 
        tk_obj = yf.Ticker(ticker)
        history = tk_obj.history(period=period, auto_adjust=False)
        if history is None or history.empty:
            raise ValueError(f"No data returned for ticker '{ticker}'. Check the symbol.")
 
        try:
            info = tk_obj.info or {}
        except Exception:
            info = {}
 
        market_history = None
        if with_market:
            try:
                market_history = yf.Ticker(MARKET_INDEX_TICKER).history(period=period)
            except Exception:
                market_history = None
 
        dataset = StockDataset(ticker=ticker, history=history, info=info,
                                market_history=market_history)
        self._cache[cache_key] = dataset
        return dataset
 
    def clear_cache(self) -> None:
        self._cache.clear()
 
 
# =================================================================================
#  ANALYSIS MODULE  (analysis/metrics.py, analysis/stats.py)
# =================================================================================
class MetricsCalculator:
    """Computes financial indicators and descriptive statistics for a dataset."""
 
    @staticmethod
    def daily_returns(close: pd.Series) -> pd.Series:
        return close.pct_change().dropna()
 
    @staticmethod
    def moving_average(close: pd.Series, window: int) -> pd.Series:
        return close.rolling(window=window).mean()
 
    @staticmethod
    def rolling_volatility(returns: pd.Series, window: int = 20) -> pd.Series:
        return returns.rolling(window=window).std() * np.sqrt(252)
 
    @staticmethod
    def rsi(close: pd.Series, period: int = 14) -> pd.Series:
        delta = close.diff()
        gain = delta.where(delta > 0, 0.0)
        loss = -delta.where(delta < 0, 0.0)
        avg_gain = gain.rolling(window=period).mean()
        avg_loss = loss.rolling(window=period).mean()
        rs = avg_gain / avg_loss.replace(0, np.nan)
        rsi_series = 100 - (100 / (1 + rs))
        return rsi_series.fillna(50)
 
    @staticmethod
    def macd(close: pd.Series, fast: int = 12, slow: int = 26, signal: int = 9):
        ema_fast = close.ewm(span=fast, adjust=False).mean()
        ema_slow = close.ewm(span=slow, adjust=False).mean()
        macd_line = ema_fast - ema_slow
        signal_line = macd_line.ewm(span=signal, adjust=False).mean()
        histogram = macd_line - signal_line
        return macd_line, signal_line, histogram
 
    @staticmethod
    def cumulative_returns(returns: pd.Series) -> pd.Series:
        return (1 + returns).cumprod() - 1
 
    @staticmethod
    def beta(stock_returns: pd.Series, market_returns: pd.Series) -> Optional[float]:
        aligned = pd.concat([stock_returns, market_returns], axis=1).dropna()
        if len(aligned) < 2:
            return None
        cov = np.cov(aligned.iloc[:, 0], aligned.iloc[:, 1])[0][1]
        var = np.var(aligned.iloc[:, 1])
        if var == 0:
            return None
        return float(cov / var)
 
    @staticmethod
    def detect_outliers_iqr(series: pd.Series) -> pd.Series:
        """Returns a boolean mask of IQR-based outliers."""
        q1, q3 = series.quantile(0.25), series.quantile(0.75)
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        return (series < lower) | (series > upper)
 
    @staticmethod
    def missing_value_report(df: pd.DataFrame) -> pd.DataFrame:
        missing = df.isna().sum()
        pct = (missing / len(df) * 100).round(2)
        return pd.DataFrame({"missing_count": missing, "missing_pct": pct})
 
    @classmethod
    def build_summary(cls, dataset: StockDataset) -> dict:
        """Compute the full metrics dictionary shown on the metrics panel."""
        hist = dataset.history
        close = hist["Close"]
        info = dataset.info or {}
 
        returns = cls.daily_returns(close)
        volatility = returns.std() * np.sqrt(252) if len(returns) > 1 else np.nan
        ma20 = cls.moving_average(close, 20)
        ma50 = cls.moving_average(close, 50)
        rsi_series = cls.rsi(close)
 
        beta_val = info.get("beta")
        if beta_val is None and dataset.market_history is not None:
            mkt_returns = cls.daily_returns(dataset.market_history["Close"])
            beta_val = cls.beta(returns, mkt_returns)
 
        summary = {
            "current_price": float(close.iloc[-1]),
            "open_price": float(hist["Open"].iloc[-1]),
            "prev_close": float(close.iloc[-2]) if len(close) > 1 else float(close.iloc[-1]),
            "close_price": float(close.iloc[-1]),
            "52w_high": float(hist["High"].max()),
            "52w_low": float(hist["Low"].min()),
            "daily_return_pct": float(returns.iloc[-1] * 100) if len(returns) else np.nan,
            "avg_daily_return_pct": float(returns.mean() * 100) if len(returns) else np.nan,
            "volatility_annualized_pct": float(volatility * 100) if not np.isnan(volatility) else np.nan,
            "ma20": float(ma20.iloc[-1]) if not np.isnan(ma20.iloc[-1]) else np.nan,
            "ma50": float(ma50.iloc[-1]) if len(ma50.dropna()) else np.nan,
            "rsi": float(rsi_series.iloc[-1]),
            "volume": int(hist["Volume"].iloc[-1]),
            "avg_volume": float(hist["Volume"].mean()),
            "dividend_yield": info.get("dividendYield"),
            "market_cap": info.get("marketCap"),
            "pe_ratio": info.get("trailingPE"),
            "beta": beta_val,
            "company_name": info.get("shortName") or info.get("longName") or dataset.ticker,
            "sector": info.get("sector", "N/A"),
            "currency": info.get("currency", "USD"),
        }
        return summary
 
 
# =================================================================================
#  CHARTS MODULE  (gui/charts/chart_factory.py)
# =================================================================================
class ChartFactory:
    """Builds styled matplotlib Figures for every visualization the app offers."""
 
    @staticmethod
    def _new_fig(figsize=(8, 4.5)) -> tuple[Figure, plt.Axes]:
        fig = Figure(figsize=figsize, dpi=100)
        ax = fig.add_subplot(111)
        fig.patch.set_facecolor(Theme.BG_CARD)
        ax.set_facecolor(Theme.BG_CARD)
        return fig, ax
 
    @classmethod
    def price_trend(cls, hist: pd.DataFrame, ticker: str) -> Figure:
        fig, ax = cls._new_fig()
        ax.plot(hist.index, hist["Close"], color=Theme.ACCENT, linewidth=1.6, label="Close")
        ax.fill_between(hist.index, hist["Close"], hist["Close"].min(),
                         color=Theme.ACCENT, alpha=0.08)
        ax.set_title(f"{ticker} — Price Trend", fontsize=13, fontweight="bold")
        ax.set_ylabel("Price")
        ax.xaxis.set_major_locator(mdates.AutoDateLocator())
        ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(ax.xaxis.get_major_locator()))
        ax.legend(frameon=False)
        fig.tight_layout()
        return fig
 
    @classmethod
    def moving_average_comparison(cls, hist: pd.DataFrame, ticker: str) -> Figure:
        fig, ax = cls._new_fig()
        ma20 = MetricsCalculator.moving_average(hist["Close"], 20)
        ma50 = MetricsCalculator.moving_average(hist["Close"], 50)
        ax.plot(hist.index, hist["Close"], color=Theme.TEXT_SECONDARY, linewidth=1.1,
                label="Close", alpha=0.8)
        ax.plot(hist.index, ma20, color=Theme.ACCENT_GREEN, linewidth=1.8, label="MA 20")
        ax.plot(hist.index, ma50, color=Theme.ACCENT_AMBER, linewidth=1.8, label="MA 50")
        ax.set_title(f"{ticker} — Moving Average Comparison", fontsize=13, fontweight="bold")
        ax.legend(frameon=False)
        fig.tight_layout()
        return fig
 
    @classmethod
    def volume_analysis(cls, hist: pd.DataFrame, ticker: str) -> Figure:
        fig, ax = cls._new_fig()
        colors = np.where(hist["Close"].diff().fillna(0) >= 0, Theme.ACCENT_GREEN, Theme.ACCENT_RED)
        ax.bar(hist.index, hist["Volume"], color=colors, width=1.0, alpha=0.85)
        ax.set_title(f"{ticker} — Trading Volume", fontsize=13, fontweight="bold")
        ax.set_ylabel("Volume")
        fig.tight_layout()
        return fig
 
    @classmethod
    def returns_histogram(cls, hist: pd.DataFrame, ticker: str) -> Figure:
        fig, ax = cls._new_fig()
        returns = MetricsCalculator.daily_returns(hist["Close"]) * 100
        sns.histplot(returns, bins=40, kde=True, ax=ax, color=Theme.ACCENT, edgecolor=None)
        ax.axvline(returns.mean(), color=Theme.ACCENT_AMBER, linestyle="--", linewidth=1.4,
                   label=f"Mean: {returns.mean():.2f}%")
        ax.set_title(f"{ticker} — Daily Returns Distribution", fontsize=13, fontweight="bold")
        ax.set_xlabel("Daily Return (%)")
        ax.legend(frameon=False)
        fig.tight_layout()
        return fig
 
    @classmethod
    def candlestick(cls, hist: pd.DataFrame, ticker: str) -> Figure:
        fig, ax = cls._new_fig()
        data = hist.tail(90).copy()
        data["idx"] = range(len(data))
        width = 0.6
        for _, row in data.iterrows():
            color = Theme.ACCENT_GREEN if row["Close"] >= row["Open"] else Theme.ACCENT_RED
            ax.plot([row["idx"], row["idx"]], [row["Low"], row["High"]],
                    color=color, linewidth=0.9)
            lower = min(row["Open"], row["Close"])
            height = abs(row["Close"] - row["Open"]) or 0.01
            ax.add_patch(Rectangle((row["idx"] - width / 2, lower), width, height,
                                    facecolor=color, edgecolor=color))
        step = max(len(data) // 8, 1)
        tick_positions = data["idx"][::step]
        tick_labels = data.index[::step].strftime("%b %d")
        ax.set_xticks(tick_positions)
        ax.set_xticklabels(tick_labels, rotation=30, ha="right")
        ax.set_title(f"{ticker} — Candlestick (Last 90 Sessions)", fontsize=13, fontweight="bold")
        fig.tight_layout()
        return fig
 
    @classmethod
    def rolling_volatility(cls, hist: pd.DataFrame, ticker: str) -> Figure:
        fig, ax = cls._new_fig()
        returns = MetricsCalculator.daily_returns(hist["Close"])
        rvol = MetricsCalculator.rolling_volatility(returns, 20) * 100
        ax.plot(rvol.index, rvol, color=Theme.ACCENT_PURPLE, linewidth=1.6)
        ax.fill_between(rvol.index, rvol, 0, color=Theme.ACCENT_PURPLE, alpha=0.1)
        ax.set_title(f"{ticker} — 20-Day Rolling Volatility (Annualized %)",
                     fontsize=13, fontweight="bold")
        fig.tight_layout()
        return fig
 
    @classmethod
    def monthly_returns(cls, hist: pd.DataFrame, ticker: str) -> Figure:
        fig, ax = cls._new_fig()
        monthly = hist["Close"].resample("ME").last().pct_change().dropna() * 100
        colors = [Theme.ACCENT_GREEN if v >= 0 else Theme.ACCENT_RED for v in monthly]
        ax.bar(monthly.index.strftime("%b '%y"), monthly.values, color=colors)
        ax.set_title(f"{ticker} — Monthly Returns (%)", fontsize=13, fontweight="bold")
        ax.tick_params(axis="x", rotation=45)
        fig.tight_layout()
        return fig
 
    @classmethod
    def correlation_heatmap(cls, price_frame: pd.DataFrame) -> Figure:
        fig, ax = cls._new_fig(figsize=(6, 5.5))
        corr = price_frame.pct_change().dropna().corr()
        sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1,
                    ax=ax, cbar_kws={"shrink": 0.8}, linewidths=0.5,
                    linecolor=Theme.BG_CARD)
        ax.set_title("Correlation Matrix", fontsize=13, fontweight="bold")
        fig.tight_layout()
        return fig
 
    @classmethod
    def risk_return_scatter(cls, price_frame: pd.DataFrame) -> Figure:
        fig, ax = cls._new_fig(figsize=(6, 5.5))
        returns = price_frame.pct_change().dropna()
        mean_ret = returns.mean() * 252 * 100
        vol = returns.std() * np.sqrt(252) * 100
        palette = sns.color_palette("husl", len(price_frame.columns))
        for i, ticker in enumerate(price_frame.columns):
            ax.scatter(vol[ticker], mean_ret[ticker], s=140, color=palette[i], label=ticker,
                       edgecolor="white", linewidth=0.6)
            ax.annotate(ticker, (vol[ticker], mean_ret[ticker]), textcoords="offset points",
                        xytext=(8, 6), fontsize=9)
        ax.set_xlabel("Annualized Volatility (%)")
        ax.set_ylabel("Annualized Return (%)")
        ax.set_title("Risk vs Return", fontsize=13, fontweight="bold")
        ax.axhline(0, color=Theme.TEXT_MUTED, linewidth=0.8)
        fig.tight_layout()
        return fig
 
    @classmethod
    def portfolio_performance(cls, price_frame: pd.DataFrame) -> Figure:
        fig, ax = cls._new_fig(figsize=(8, 4.5))
        cum_returns = (price_frame.pct_change().fillna(0) + 1).cumprod() - 1
        palette = sns.color_palette("husl", len(price_frame.columns))
        for i, ticker in enumerate(price_frame.columns):
            ax.plot(cum_returns.index, cum_returns[ticker] * 100, label=ticker,
                    color=palette[i], linewidth=1.6)
        ax.set_title("Cumulative Performance Comparison (%)", fontsize=13, fontweight="bold")
        ax.legend(frameon=False)
        fig.tight_layout()
        return fig
 
    @classmethod
    def portfolio_allocation(cls, weights: dict) -> Figure:
        fig, ax = cls._new_fig(figsize=(5.5, 5.5))
        labels = list(weights.keys())
        sizes = list(weights.values())
        palette = sns.color_palette("husl", len(labels))
        wedges, _texts, autotexts = ax.pie(
            sizes, labels=labels, autopct="%1.1f%%", colors=palette,
            wedgeprops={"edgecolor": Theme.BG_CARD, "linewidth": 2},
            textprops={"color": Theme.TEXT_PRIMARY},
        )
        for at in autotexts:
            at.set_color("#0b1220")
            at.set_fontweight("bold")
        ax.set_title("Portfolio Allocation", fontsize=13, fontweight="bold")
        fig.tight_layout()
        return fig
 
 
# =================================================================================
#  REPORTS MODULE  (reports/excel_exporter.py)
# =================================================================================
class ExcelExporter:
    """Generates formatted, multi-sheet Excel reports using openpyxl."""
 
    HEADER_FILL = PatternFill(start_color="1F2937", end_color="1F2937", fill_type="solid")
    HEADER_FONT = Font(color="FFFFFF", bold=True, size=11)
    TITLE_FONT = Font(color="1F2937", bold=True, size=16)
    THIN_BORDER = Border(*(Side(style="thin", color="D1D5DB"),) * 4)
 
    @classmethod
    def export_single_stock(cls, path: str, dataset: StockDataset, summary: dict) -> None:
        wb = Workbook()
 
        # --- Summary sheet ---------------------------------------------------
        ws = wb.active
        ws.title = "Summary"
        ws["B2"] = f"{summary.get('company_name', dataset.ticker)} ({dataset.ticker})"
        ws["B2"].font = cls.TITLE_FONT
        ws["B3"] = f"Report generated: {datetime.now():%Y-%m-%d %H:%M}"
        ws["B3"].font = Font(italic=True, color="6B7280")
 
        rows = [
            ("Current Price", fmt_num(summary.get("current_price"))),
            ("Open Price", fmt_num(summary.get("open_price"))),
            ("52-Week High", fmt_num(summary.get("52w_high"))),
            ("52-Week Low", fmt_num(summary.get("52w_low"))),
            ("Daily Return (%)", fmt_num(summary.get("daily_return_pct"))),
            ("Avg Daily Return (%)", fmt_num(summary.get("avg_daily_return_pct"))),
            ("Volatility (Annualized %)", fmt_num(summary.get("volatility_annualized_pct"))),
            ("20-Day MA", fmt_num(summary.get("ma20"))),
            ("50-Day MA", fmt_num(summary.get("ma50"))),
            ("RSI", fmt_num(summary.get("rsi"))),
            ("Volume", fmt_large_number(summary.get("volume"))),
            ("Avg Volume", fmt_large_number(summary.get("avg_volume"))),
            ("Market Cap", fmt_large_number(summary.get("market_cap"))),
            ("P/E Ratio", fmt_num(summary.get("pe_ratio"))),
            ("Beta", fmt_num(summary.get("beta"))),
            ("Sector", summary.get("sector", "N/A")),
        ]
        start_row = 5
        ws.cell(row=start_row, column=2, value="Metric").font = cls.HEADER_FONT
        ws.cell(row=start_row, column=2).fill = cls.HEADER_FILL
        ws.cell(row=start_row, column=3, value="Value").font = cls.HEADER_FONT
        ws.cell(row=start_row, column=3).fill = cls.HEADER_FILL
        for i, (label, value) in enumerate(rows, start=start_row + 1):
            ws.cell(row=i, column=2, value=label).border = cls.THIN_BORDER
            ws.cell(row=i, column=3, value=value).border = cls.THIN_BORDER
        cls._autosize(ws)
 
        # --- Historical Data sheet --------------------------------------------
        ws2 = wb.create_sheet("Historical Data")
        hist = dataset.history.reset_index()
        hist_cols = [c for c in ["Date", "Open", "High", "Low", "Close", "Volume"] if c in hist.columns]
        hist = hist[hist_cols]
        hist["Daily Return %"] = hist["Close"].pct_change() * 100
 
        for c_idx, col_name in enumerate(hist.columns, start=1):
            cell = ws2.cell(row=1, column=c_idx, value=col_name)
            cell.font = cls.HEADER_FONT
            cell.fill = cls.HEADER_FILL
        for r_idx, row in enumerate(hist.itertuples(index=False), start=2):
            for c_idx, value in enumerate(row, start=1):
                if hasattr(value, "strftime"):
                    value = value.strftime("%Y-%m-%d")
                ws2.cell(row=r_idx, column=c_idx, value=value)
 
        return_col_letter = get_column_letter(hist.columns.get_loc("Daily Return %") + 1)
        last_row = len(hist) + 1
        color_rule = ColorScaleRule(
            start_type="min", start_color="F87171",
            mid_type="percentile", mid_value=50, mid_color="FDE68A",
            end_type="max", end_color="86EFAC",
        )
        ws2.conditional_formatting.add(
            f"{return_col_letter}2:{return_col_letter}{last_row}", color_rule
        )
        cls._autosize(ws2)
 
        # --- Embedded chart ----------------------------------------------------
        close_col = hist.columns.get_loc("Close") + 1
        chart = LineChart()
        chart.title = "Closing Price"
        chart.y_axis.title = "Price"
        chart.x_axis.title = "Date"
        data_ref = Reference(ws2, min_col=close_col, min_row=1, max_row=last_row)
        chart.add_data(data_ref, titles_from_data=True)
        chart.height, chart.width = 9, 20
        ws2.add_chart(chart, f"I2")
 
        wb.save(path)
 
    @classmethod
    def export_portfolio(cls, path: str, price_frame: pd.DataFrame,
                          metrics_frame: pd.DataFrame) -> None:
        wb = Workbook()
        ws = wb.active
        ws.title = "Portfolio Metrics"
        ws["B2"] = "Portfolio Comparison Report"
        ws["B2"].font = cls.TITLE_FONT
        ws["B3"] = f"Generated: {datetime.now():%Y-%m-%d %H:%M}"
 
        start_row = 5
        headers = ["Ticker"] + list(metrics_frame.columns)
        for c_idx, header in enumerate(headers, start=2):
            cell = ws.cell(row=start_row, column=c_idx, value=header)
            cell.font = cls.HEADER_FONT
            cell.fill = cls.HEADER_FILL
        for r_idx, (ticker, row) in enumerate(metrics_frame.iterrows(), start=start_row + 1):
            ws.cell(row=r_idx, column=2, value=ticker).border = cls.THIN_BORDER
            for c_idx, value in enumerate(row, start=3):
                ws.cell(row=r_idx, column=c_idx, value=round(float(value), 4) if pd.notna(value) else "N/A").border = cls.THIN_BORDER
        cls._autosize(ws)
 
        ws2 = wb.create_sheet("Price History")
        prices = price_frame.reset_index()
        for c_idx, col in enumerate(prices.columns, start=1):
            cell = ws2.cell(row=1, column=c_idx, value=str(col))
            cell.font = cls.HEADER_FONT
            cell.fill = cls.HEADER_FILL
        for r_idx, row in enumerate(prices.itertuples(index=False), start=2):
            for c_idx, value in enumerate(row, start=1):
                if hasattr(value, "strftime"):
                    value = value.strftime("%Y-%m-%d")
                ws2.cell(row=r_idx, column=c_idx, value=value)
        cls._autosize(ws2)
 
        ws3 = wb.create_sheet("Correlation Matrix")
        corr = price_frame.pct_change().dropna().corr()
        ws3.cell(row=1, column=1, value="")
        for i, col in enumerate(corr.columns, start=2):
            ws3.cell(row=1, column=i, value=col).font = cls.HEADER_FONT
            ws3.cell(row=1, column=i).fill = cls.HEADER_FILL
        for r_idx, (idx, row) in enumerate(corr.iterrows(), start=2):
            ws3.cell(row=r_idx, column=1, value=idx).font = cls.HEADER_FONT
            ws3.cell(row=r_idx, column=1).fill = cls.HEADER_FILL
            for c_idx, value in enumerate(row, start=2):
                ws3.cell(row=r_idx, column=c_idx, value=round(float(value), 3))
        rule = ColorScaleRule(start_type="min", start_color="F87171",
                               mid_type="num", mid_value=0, mid_color="FDE68A",
                               end_type="max", end_color="86EFAC")
        last_col_letter = get_column_letter(len(corr.columns) + 1)
        ws3.conditional_formatting.add(f"B2:{last_col_letter}{len(corr)+1}", rule)
        cls._autosize(ws3)
 
        wb.save(path)
 
    @staticmethod
    def _autosize(ws) -> None:
        for column_cells in ws.columns:
            length = max((len(str(c.value)) for c in column_cells if c.value is not None), default=8)
            col_letter = get_column_letter(column_cells[0].column)
            ws.column_dimensions[col_letter].width = min(max(length + 3, 10), 45)
 
 
# =================================================================================
#  WIDGETS MODULE  (gui/widgets/*)
# =================================================================================
class RoundedButton(tk.Canvas):
    """A canvas-drawn button with rounded corners and a hover effect."""
 
    def __init__(self, parent, text: str, command: Optional[Callable] = None,
                 bg=Theme.ACCENT, hover_bg=Theme.ACCENT_HOVER, fg=Theme.TEXT_ON_ACCENT,
                 width: int = 150, height: int = 38, radius: int = 12,
                 font=Theme.FONT_BODY_BOLD, **kwargs):
        super().__init__(parent, width=width, height=height, bg=parent["bg"],
                          highlightthickness=0, **kwargs)
        self.command = command
        self.bg_color = bg
        self.hover_color = hover_bg
        self.fg_color = fg
        self.radius = radius
        self.text = text
        self.font = font
        self.width = width
        self.height = height
        self._draw(bg)
        self.bind("<Enter>", lambda e: self._draw(self.hover_color))
        self.bind("<Leave>", lambda e: self._draw(self.bg_color))
        self.bind("<Button-1>", self._on_click)
 
    def _round_rect(self, x1, y1, x2, y2, r, **kw):
        points = [x1 + r, y1, x2 - r, y1, x2, y1, x2, y1 + r, x2, y2 - r, x2, y2,
                  x2 - r, y2, x1 + r, y2, x1, y2, x1, y2 - r, x1, y1 + r, x1, y1]
        return self.create_polygon(points, smooth=True, **kw)
 
    def _draw(self, color) -> None:
        self.delete("all")
        self._round_rect(2, 2, self.width - 2, self.height - 2, self.radius, fill=color, outline="")
        self.create_text(self.width / 2, self.height / 2, text=self.text,
                          fill=self.fg_color, font=self.font)
 
    def _on_click(self, _event) -> None:
        if self.command:
            self.command()
 
    def set_enabled(self, enabled: bool) -> None:
        if enabled:
            self.bind("<Button-1>", self._on_click)
            self._draw(self.bg_color)
        else:
            self.unbind("<Button-1>")
            self._draw(Theme.TEXT_MUTED)
 
 
class Card(tk.Frame):
    """A simple elevated 'card' container with consistent padding."""
 
    def __init__(self, parent, **kwargs):
        super().__init__(parent, bg=Theme.BG_CARD, highlightbackground=Theme.BORDER,
                          highlightthickness=1, **kwargs)
 
 
class MetricCard(Card):
    """Small dashboard tile: label on top, big value below, optional delta color."""
 
    def __init__(self, parent, label: str, value: str = "—", accent: str = Theme.TEXT_PRIMARY):
        super().__init__(parent)
        self.label_var = tk.StringVar(value=label)
        self.value_var = tk.StringVar(value=value)
        tk.Label(self, textvariable=self.label_var, font=Theme.FONT_METRIC_LABEL,
                 fg=Theme.TEXT_SECONDARY, bg=Theme.BG_CARD).pack(anchor="w", padx=12, pady=(10, 0))
        self.value_label = tk.Label(self, textvariable=self.value_var, font=Theme.FONT_METRIC_VALUE,
                                     fg=accent, bg=Theme.BG_CARD)
        self.value_label.pack(anchor="w", padx=12, pady=(0, 10))
 
    def update_value(self, value: str, accent: Optional[str] = None) -> None:
        self.value_var.set(value)
        if accent:
            self.value_label.configure(fg=accent)
 
 
class StatusBar(tk.Frame):
    """Bottom status bar with message + indeterminate progress indicator."""
 
    def __init__(self, parent):
        super().__init__(parent, bg=Theme.BG_SIDEBAR, height=30)
        self.pack_propagate(False)
        self.status_var = tk.StringVar(value="Ready")
        tk.Label(self, textvariable=self.status_var, bg=Theme.BG_SIDEBAR,
                 fg=Theme.TEXT_SECONDARY, font=Theme.FONT_SMALL, anchor="w").pack(
            side="left", padx=12)
 
        style = ttk.Style()
        style.configure("Status.Horizontal.TProgressbar", troughcolor=Theme.BG_SIDEBAR,
                        background=Theme.ACCENT, bordercolor=Theme.BG_SIDEBAR,
                        lightcolor=Theme.ACCENT, darkcolor=Theme.ACCENT)
        self.progress = ttk.Progressbar(self, mode="indeterminate", length=140,
                                         style="Status.Horizontal.TProgressbar")
        self.progress.pack(side="right", padx=12, pady=5)
 
        self.clock_var = tk.StringVar()
        tk.Label(self, textvariable=self.clock_var, bg=Theme.BG_SIDEBAR,
                 fg=Theme.TEXT_MUTED, font=Theme.FONT_SMALL).pack(side="right", padx=12)
        self._tick()
 
    def _tick(self) -> None:
        self.clock_var.set(datetime.now().strftime("%Y-%m-%d  %H:%M:%S"))
        self.after(1000, self._tick)
 
    def set_status(self, text: str) -> None:
        self.status_var.set(text)
 
    def start_progress(self, text: str = "Loading...") -> None:
        self.set_status(text)
        self.progress.start(12)
 
    def stop_progress(self, text: str = "Ready") -> None:
        self.progress.stop()
        self.set_status(text)
 
 
class SplashScreen(tk.Toplevel):
    """Startup splash screen shown briefly while the main app initializes."""
 
    def __init__(self, master, on_finish: Callable):
        super().__init__(master)
        self.overrideredirect(True)
        self.configure(bg=Theme.BG_PRIMARY)
        width, height = 480, 300
        self._center(width, height)
 
        wrapper = tk.Frame(self, bg=Theme.BG_PRIMARY, highlightbackground=Theme.ACCENT,
                            highlightthickness=1)
        wrapper.pack(fill="both", expand=True)
 
        tk.Label(wrapper, text="\U0001F4C8", font=(Theme.FONT_FAMILY, 46), bg=Theme.BG_PRIMARY,
                 fg=Theme.ACCENT).pack(pady=(50, 6))
        tk.Label(wrapper, text="STOCK MARKET ANALYZER", font=Theme.FONT_TITLE,
                 bg=Theme.BG_PRIMARY, fg=Theme.TEXT_PRIMARY).pack()
        tk.Label(wrapper, text="Financial Data Analytics Desktop Suite", font=Theme.FONT_SUBTITLE,
                 bg=Theme.BG_PRIMARY, fg=Theme.TEXT_SECONDARY).pack(pady=(2, 30))
 
        style = ttk.Style()
        style.configure("Splash.Horizontal.TProgressbar", troughcolor=Theme.BG_CARD,
                        background=Theme.ACCENT, bordercolor=Theme.BG_PRIMARY)
        pb = ttk.Progressbar(wrapper, mode="indeterminate", length=320,
                              style="Splash.Horizontal.TProgressbar")
        pb.pack()
        pb.start(10)
 
        self.after(1600, lambda: self._close(on_finish))
 
    def _center(self, width, height) -> None:
        sw = self.winfo_screenwidth()
        sh = self.winfo_screenheight()
        x, y = (sw - width) // 2, (sh - height) // 2
        self.geometry(f"{width}x{height}+{x}+{y}")
 
    def _close(self, on_finish: Callable) -> None:
        self.destroy()
        on_finish()
 
 
class Sidebar(tk.Frame):
    """Left navigation rail with icon-style buttons and hover highlighting."""
 
    NAV_ITEMS = [
        ("dashboard", "\u25A3  Dashboard"),
        ("analysis", "\U0001F4C8  Stock Analysis"),
        ("portfolio", "\u2696  Portfolio Compare"),
        ("reports", "\U0001F4C4  Excel Reports"),
        ("watchlist", "\u2605  Watchlist & History"),
    ]
 
    def __init__(self, parent, on_navigate: Callable[[str], None]):
        super().__init__(parent, bg=Theme.BG_SIDEBAR, width=Theme.SIDEBAR_WIDTH)
        self.pack_propagate(False)
        self.on_navigate = on_navigate
        self.buttons: dict[str, tk.Label] = {}
        self.active_key = "dashboard"
 
        header = tk.Frame(self, bg=Theme.BG_SIDEBAR)
        header.pack(fill="x", pady=(24, 30), padx=18)
        tk.Label(header, text="\U0001F4C8", font=(Theme.FONT_FAMILY, 22), bg=Theme.BG_SIDEBAR,
                 fg=Theme.ACCENT).pack(side="left")
        tk.Label(header, text=" StockAnalyzer", font=(Theme.FONT_FAMILY, 14, "bold"),
                 bg=Theme.BG_SIDEBAR, fg=Theme.TEXT_PRIMARY).pack(side="left")
 
        for key, label in self.NAV_ITEMS:
            self._add_nav_button(key, label)
 
        tk.Frame(self, bg=Theme.BORDER, height=1).pack(fill="x", padx=18, pady=14)
        tk.Label(self, text="v1.0.0  \u2022  Dark Mode", bg=Theme.BG_SIDEBAR,
                 fg=Theme.TEXT_MUTED, font=Theme.FONT_SMALL).pack(side="bottom", pady=16)
 
    def _add_nav_button(self, key: str, label: str) -> None:
        btn = tk.Label(self, text=label, bg=Theme.BG_SIDEBAR, fg=Theme.TEXT_SECONDARY,
                        font=Theme.FONT_BODY, anchor="w", padx=20, pady=12, cursor="hand2")
        btn.pack(fill="x", padx=10, pady=2)
        btn.bind("<Enter>", lambda e, b=btn: b.configure(bg=Theme.BG_CARD_HOVER) if key != self.active_key else None)
        btn.bind("<Leave>", lambda e, b=btn: b.configure(bg=Theme.BG_SIDEBAR) if key != self.active_key else None)
        btn.bind("<Button-1>", lambda e, k=key: self._select(k))
        self.buttons[key] = btn
 
    def _select(self, key: str) -> None:
        self.active_key = key
        for k, btn in self.buttons.items():
            if k == key:
                btn.configure(bg=Theme.ACCENT, fg=Theme.TEXT_ON_ACCENT, font=Theme.FONT_BODY_BOLD)
            else:
                btn.configure(bg=Theme.BG_SIDEBAR, fg=Theme.TEXT_SECONDARY, font=Theme.FONT_BODY)
        self.on_navigate(key)
 
    def select(self, key: str) -> None:
        self._select(key)
 
 
def styled_entry(parent, textvariable=None, width=22) -> tk.Entry:
    return tk.Entry(parent, textvariable=textvariable, width=width, bg=Theme.BG_INPUT,
                     fg=Theme.TEXT_PRIMARY, insertbackground=Theme.TEXT_PRIMARY,
                     relief="flat", font=Theme.FONT_BODY, highlightthickness=1,
                     highlightbackground=Theme.BORDER, highlightcolor=Theme.ACCENT)
 
 
def section_title(parent, text: str) -> tk.Label:
    return tk.Label(parent, text=text, font=Theme.FONT_H1, bg=Theme.BG_PRIMARY,
                     fg=Theme.TEXT_PRIMARY, anchor="w")
 
 
class ChartTab(tk.Frame):
    """A notebook tab that hosts a matplotlib figure with toolbar + PNG export."""
 
    def __init__(self, parent, figure_factory: Callable[[], Figure], title: str):
        super().__init__(parent, bg=Theme.BG_CARD)
        self.figure_factory = figure_factory
        self.title = title
        self.canvas: Optional[FigureCanvasTkAgg] = None
        self.fig: Optional[Figure] = None
        self._build_toolbar()
        self.body = tk.Frame(self, bg=Theme.BG_CARD)
        self.body.pack(fill="both", expand=True)
        self.placeholder = tk.Label(self.body, text="Run an analysis to view this chart",
                                     bg=Theme.BG_CARD, fg=Theme.TEXT_MUTED, font=Theme.FONT_BODY)
        self.placeholder.pack(expand=True)
 
    def _build_toolbar(self) -> None:
        bar = tk.Frame(self, bg=Theme.BG_CARD)
        bar.pack(fill="x", padx=8, pady=(8, 0))
        export_btn = RoundedButton(bar, "Export PNG", command=self.export_png, width=120, height=30,
                                    bg=Theme.BG_CARD_HOVER, hover_bg=Theme.ACCENT,
                                    font=Theme.FONT_SMALL)
        export_btn.pack(side="right")
 
    def render(self) -> None:
        for w in self.body.winfo_children():
            w.destroy()
        try:
            self.fig = self.figure_factory()
        except Exception as exc:
            tk.Label(self.body, text=f"Chart unavailable: {exc}", bg=Theme.BG_CARD,
                     fg=Theme.ACCENT_RED, wraplength=500).pack(expand=True)
            return
        self.canvas = FigureCanvasTkAgg(self.fig, master=self.body)
        self.canvas.draw()
        toolbar_frame = tk.Frame(self.body, bg=Theme.BG_CARD)
        toolbar_frame.pack(fill="x")
        toolbar = NavigationToolbar2Tk(self.canvas, toolbar_frame)
        toolbar.config(background=Theme.BG_CARD)
        toolbar._message_label.config(background=Theme.BG_CARD, foreground=Theme.TEXT_SECONDARY)
        toolbar.update()
        self.canvas.get_tk_widget().pack(fill="both", expand=True)
 
    def export_png(self) -> None:
        if self.fig is None:
            messagebox.showinfo("Export", "Nothing to export yet — run an analysis first.")
            return
        path = filedialog.asksaveasfilename(
            defaultextension=".png", initialfile=f"{self.title.replace(' ', '_')}.png",
            filetypes=[("PNG Image", "*.png")]
        )
        if path:
            self.fig.savefig(path, dpi=200, facecolor=Theme.BG_CARD)
            messagebox.showinfo("Export Complete", f"Chart saved to:\n{path}")
 
 
# =================================================================================
#  PAGES MODULE  (gui/pages/*)
# =================================================================================
class BasePage(tk.Frame):
    """Common base for all navigable pages."""
 
    def __init__(self, parent, app: "StockAnalyzerApp"):
        super().__init__(parent, bg=Theme.BG_PRIMARY)
        self.app = app
 
 
class DashboardPage(BasePage):
    def __init__(self, parent, app):
        super().__init__(parent, app)
        self._build()
 
    def _build(self):
        section_title(self, "Dashboard").pack(anchor="w", padx=28, pady=(24, 4))
        tk.Label(self, text="Search any ticker to begin a full financial analysis.",
                 font=Theme.FONT_SUBTITLE, bg=Theme.BG_PRIMARY, fg=Theme.TEXT_SECONDARY,
                 anchor="w").pack(anchor="w", padx=28, pady=(0, 20))
 
        search_row = tk.Frame(self, bg=Theme.BG_PRIMARY)
        search_row.pack(fill="x", padx=28)
        self.ticker_var = tk.StringVar()
        entry = styled_entry(search_row, textvariable=self.ticker_var, width=28)
        entry.pack(side="left", ipady=6)
        entry.bind("<Return>", lambda e: self._search())
        Tooltip(entry, "Enter a ticker symbol e.g. AAPL, TSLA, INFY.NS")
 
        self.period_var = tk.StringVar(value="1 Year")
        period_menu = ttk.Combobox(search_row, textvariable=self.period_var,
                                    values=list(PERIOD_OPTIONS.keys()), state="readonly", width=12)
        period_menu.pack(side="left", padx=10)
 
        search_btn = RoundedButton(search_row, "Analyze", command=self._search, width=120)
        search_btn.pack(side="left", padx=6)
 
        self.app.root.bind("<Control-f>", lambda e: entry.focus_set())
        self.app.root.bind("<F5>", lambda e: self._search())
 
        # Quick metric cards
        self.cards_frame = tk.Frame(self, bg=Theme.BG_PRIMARY)
        self.cards_frame.pack(fill="x", padx=28, pady=26)
        self.cards: dict[str, MetricCard] = {}
        card_defs = [
            ("current_price", "CURRENT PRICE"), ("daily_return_pct", "DAILY RETURN"),
            ("52w_high", "52-WEEK HIGH"), ("52w_low", "52-WEEK LOW"),
            ("volatility_annualized_pct", "VOLATILITY (ANN.)"), ("rsi", "RSI (14)"),
        ]
        for i, (key, label) in enumerate(card_defs):
            card = MetricCard(self.cards_frame, label)
            card.grid(row=0, column=i, padx=8, sticky="nsew")
            self.cards_frame.columnconfigure(i, weight=1)
            self.cards[key] = card
 
        lists_row = tk.Frame(self, bg=Theme.BG_PRIMARY)
        lists_row.pack(fill="both", expand=True, padx=28, pady=(0, 20))
 
        recent_card = Card(lists_row)
        recent_card.pack(side="left", fill="both", expand=True, padx=(0, 10))
        tk.Label(recent_card, text="Recent Searches", font=Theme.FONT_H2, bg=Theme.BG_CARD,
                 fg=Theme.TEXT_PRIMARY).pack(anchor="w", padx=14, pady=(12, 6))
        self.recent_list = tk.Listbox(recent_card, bg=Theme.BG_INPUT, fg=Theme.TEXT_PRIMARY,
                                       relief="flat", highlightthickness=0, font=Theme.FONT_BODY,
                                       activestyle="none")
        self.recent_list.pack(fill="both", expand=True, padx=14, pady=(0, 14))
        self.recent_list.bind("<Double-Button-1>", self._load_from_recent)
 
        watch_card = Card(lists_row)
        watch_card.pack(side="left", fill="both", expand=True, padx=(10, 0))
        tk.Label(watch_card, text="Watchlist", font=Theme.FONT_H2, bg=Theme.BG_CARD,
                 fg=Theme.TEXT_PRIMARY).pack(anchor="w", padx=14, pady=(12, 6))
        self.watch_list = tk.Listbox(watch_card, bg=Theme.BG_INPUT, fg=Theme.ACCENT_AMBER,
                                      relief="flat", highlightthickness=0, font=Theme.FONT_BODY,
                                      activestyle="none")
        self.watch_list.pack(fill="both", expand=True, padx=14, pady=(0, 14))
        self.watch_list.bind("<Double-Button-1>", self._load_from_watchlist)
 
        self.refresh_lists()
 
    def refresh_lists(self):
        self.recent_list.delete(0, tk.END)
        for t in self.app.storage.data["recent_searches"]:
            self.recent_list.insert(tk.END, f"  {t}")
        self.watch_list.delete(0, tk.END)
        for t in self.app.storage.data["watchlist"]:
            self.watch_list.insert(tk.END, f"  \u2605 {t}")
 
    def _load_from_recent(self, _event):
        sel = self.recent_list.curselection()
        if sel:
            ticker = self.recent_list.get(sel[0]).strip()
            self.ticker_var.set(ticker)
            self._search()
 
    def _load_from_watchlist(self, _event):
        sel = self.watch_list.curselection()
        if sel:
            ticker = self.watch_list.get(sel[0]).replace("\u2605", "").strip()
            self.ticker_var.set(ticker)
            self._search()
 
    def _search(self):
        ticker = validate_ticker(self.ticker_var.get())
        if not ticker:
            messagebox.showwarning("Invalid Ticker", "Please enter a valid ticker symbol (e.g. AAPL).")
            return
        period = PERIOD_OPTIONS[self.period_var.get()]
        self.app.load_ticker(ticker, period, on_dashboard_update=self.update_cards)
        self.app.sidebar.select("analysis")
 
    def update_cards(self, summary: dict):
        self.cards["current_price"].update_value(fmt_num(summary.get("current_price"), 2, "  " + summary.get("currency", "")))
        ret = summary.get("daily_return_pct")
        color = Theme.ACCENT_GREEN if (ret or 0) >= 0 else Theme.ACCENT_RED
        self.cards["daily_return_pct"].update_value(fmt_num(ret, 2, "%"), accent=color)
        self.cards["52w_high"].update_value(fmt_num(summary.get("52w_high")))
        self.cards["52w_low"].update_value(fmt_num(summary.get("52w_low")))
        self.cards["volatility_annualized_pct"].update_value(fmt_num(summary.get("volatility_annualized_pct"), 2, "%"))
        rsi_val = summary.get("rsi")
        rsi_color = Theme.ACCENT_RED if (rsi_val or 50) > 70 else (Theme.ACCENT_GREEN if (rsi_val or 50) < 30 else Theme.TEXT_PRIMARY)
        self.cards["rsi"].update_value(fmt_num(rsi_val, 1), accent=rsi_color)
        self.refresh_lists()
 
 
class AnalysisPage(BasePage):
    """Deep-dive page: full metrics table + tabbed chart gallery for one ticker."""
 
    CHART_DEFS = [
        ("Price Trend", "price_trend"),
        ("Moving Averages", "moving_average_comparison"),
        ("Volume", "volume_analysis"),
        ("Returns Histogram", "returns_histogram"),
        ("Candlestick", "candlestick"),
        ("Rolling Volatility", "rolling_volatility"),
        ("Monthly Returns", "monthly_returns"),
    ]
 
    def __init__(self, parent, app):
        super().__init__(parent, app)
        self.dataset: Optional[StockDataset] = None
        self.summary: Optional[dict] = None
        self._build()
 
    def _build(self):
        header = tk.Frame(self, bg=Theme.BG_PRIMARY)
        header.pack(fill="x", padx=28, pady=(24, 6))
        self.title_var = tk.StringVar(value="Stock Analysis")
        tk.Label(header, textvariable=self.title_var, font=Theme.FONT_H1, bg=Theme.BG_PRIMARY,
                 fg=Theme.TEXT_PRIMARY).pack(side="left")
        self.watch_btn = RoundedButton(header, "\u2606 Add to Watchlist", command=self._toggle_watch,
                                        width=190, height=32, bg=Theme.BG_CARD_HOVER,
                                        hover_bg=Theme.ACCENT_AMBER, font=Theme.FONT_SMALL)
        self.watch_btn.pack(side="right")
 
        body = tk.Frame(self, bg=Theme.BG_PRIMARY)
        body.pack(fill="both", expand=True, padx=28, pady=(10, 20))
 
        # Left: metrics table
        metrics_card = Card(body, width=280)
        metrics_card.pack(side="left", fill="y", padx=(0, 16))
        metrics_card.pack_propagate(False)
        tk.Label(metrics_card, text="Key Metrics", font=Theme.FONT_H2, bg=Theme.BG_CARD,
                 fg=Theme.TEXT_PRIMARY).pack(anchor="w", padx=14, pady=(14, 6))
        self.metrics_tree = ttk.Treeview(metrics_card, columns=("value",), show="tree",
                                          height=22)
        self.metrics_tree.pack(fill="both", expand=True, padx=10, pady=(0, 10))
        self._style_treeview()
 
        # Right: chart notebook
        right = tk.Frame(body, bg=Theme.BG_PRIMARY)
        right.pack(side="left", fill="both", expand=True)
        self.notebook = ttk.Notebook(right)
        self.notebook.pack(fill="both", expand=True)
        self._style_notebook()
 
        self.tabs: dict[str, ChartTab] = {}
        for label, factory_name in self.CHART_DEFS:
            tab = ChartTab(self.notebook, lambda n=factory_name: self._make_chart(n), label)
            self.notebook.add(tab, text=label)
            self.tabs[factory_name] = tab
        self.notebook.bind("<<NotebookTabChanged>>", self._on_tab_changed)
 
    def _style_treeview(self):
        style = ttk.Style()
        style.theme_use("default")
        style.configure("Treeview", background=Theme.BG_CARD, foreground=Theme.TEXT_PRIMARY,
                        fieldbackground=Theme.BG_CARD, borderwidth=0, font=Theme.FONT_BODY,
                        rowheight=26)
        style.map("Treeview", background=[("selected", Theme.ACCENT)])
 
    def _style_notebook(self):
        style = ttk.Style()
        style.configure("TNotebook", background=Theme.BG_PRIMARY, borderwidth=0)
        style.configure("TNotebook.Tab", background=Theme.BG_CARD, foreground=Theme.TEXT_SECONDARY,
                        padding=[14, 8], font=Theme.FONT_BODY)
        style.map("TNotebook.Tab", background=[("selected", Theme.ACCENT)],
                  foreground=[("selected", Theme.TEXT_ON_ACCENT)])
 
    def _make_chart(self, factory_name: str) -> Figure:
        if self.dataset is None:
            raise RuntimeError("No data loaded")
        factory = getattr(ChartFactory, factory_name)
        return factory(self.dataset.history, self.dataset.ticker)
 
    def _on_tab_changed(self, _event):
        idx = self.notebook.index(self.notebook.select())
        _, factory_name = self.CHART_DEFS[idx]
        tab = self.tabs[factory_name]
        if self.dataset is not None:
            tab.render()
 
    def load(self, dataset: StockDataset, summary: dict):
        self.dataset = dataset
        self.summary = summary
        self.title_var.set(f"{summary.get('company_name', dataset.ticker)}  ({dataset.ticker})")
        self._populate_metrics(summary)
        self._update_watch_button()
        # Render currently visible tab immediately, others lazily on click.
        current = self.notebook.select()
        if current:
            self._on_tab_changed(None)
        else:
            self.tabs["price_trend"].render()
 
    def _populate_metrics(self, summary: dict):
        for item in self.metrics_tree.get_children():
            self.metrics_tree.delete(item)
        rows = [
            ("Current Price", fmt_num(summary.get("current_price"))),
            ("Open Price", fmt_num(summary.get("open_price"))),
            ("Prev Close", fmt_num(summary.get("prev_close"))),
            ("52-Week High", fmt_num(summary.get("52w_high"))),
            ("52-Week Low", fmt_num(summary.get("52w_low"))),
            ("Daily Return", fmt_num(summary.get("daily_return_pct"), 2, "%")),
            ("Avg Daily Return", fmt_num(summary.get("avg_daily_return_pct"), 3, "%")),
            ("Volatility (Ann.)", fmt_num(summary.get("volatility_annualized_pct"), 2, "%")),
            ("MA (20-day)", fmt_num(summary.get("ma20"))),
            ("MA (50-day)", fmt_num(summary.get("ma50"))),
            ("RSI (14)", fmt_num(summary.get("rsi"), 1)),
            ("Volume", fmt_large_number(summary.get("volume"))),
            ("Avg Volume", fmt_large_number(summary.get("avg_volume"))),
            ("Dividend Yield", fmt_num((summary.get("dividend_yield") or 0) * 100, 2, "%")
                if summary.get("dividend_yield") else "N/A"),
            ("Market Cap", fmt_large_number(summary.get("market_cap"))),
            ("P/E Ratio", fmt_num(summary.get("pe_ratio"))),
            ("Beta", fmt_num(summary.get("beta"))),
            ("Sector", summary.get("sector", "N/A")),
        ]
        for label, value in rows:
            self.metrics_tree.insert("", "end", text=f"{label}:  {value}")
 
    def _toggle_watch(self):
        if self.dataset is None:
            return
        now_watchlisted = self.app.storage.toggle_watchlist(self.dataset.ticker)
        self._update_watch_button(now_watchlisted)
        self.app.dashboard_page.refresh_lists()
 
    def _update_watch_button(self, watched: Optional[bool] = None):
        if self.dataset is None:
            return
        if watched is None:
            watched = self.app.storage.is_watchlisted(self.dataset.ticker)
        self.watch_btn.text = "\u2605 In Watchlist" if watched else "\u2606 Add to Watchlist"
        self.watch_btn._draw(self.watch_btn.bg_color)
 
 
class PortfolioPage(BasePage):
    def __init__(self, parent, app):
        super().__init__(parent, app)
        self.price_frame: Optional[pd.DataFrame] = None
        self.metrics_frame: Optional[pd.DataFrame] = None
        self._build()
 
    def _build(self):
        section_title(self, "Portfolio Comparison").pack(anchor="w", padx=28, pady=(24, 4))
        tk.Label(self, text="Compare multiple tickers side-by-side (comma-separated).",
                 font=Theme.FONT_SUBTITLE, bg=Theme.BG_PRIMARY, fg=Theme.TEXT_SECONDARY
                 ).pack(anchor="w", padx=28, pady=(0, 14))
 
        row = tk.Frame(self, bg=Theme.BG_PRIMARY)
        row.pack(fill="x", padx=28)
        self.tickers_var = tk.StringVar(value="AAPL, MSFT, GOOGL, TSLA")
        entry = styled_entry(row, textvariable=self.tickers_var, width=44)
        entry.pack(side="left", ipady=6)
        self.period_var = tk.StringVar(value="1 Year")
        ttk.Combobox(row, textvariable=self.period_var, values=list(PERIOD_OPTIONS.keys()),
                     state="readonly", width=12).pack(side="left", padx=10)
        RoundedButton(row, "Compare", command=self._compare, width=120).pack(side="left")
        RoundedButton(row, "Export to Excel", command=self._export, width=150,
                      bg=Theme.BG_CARD_HOVER, hover_bg=Theme.ACCENT_GREEN).pack(side="left", padx=8)
 
        self.notebook = ttk.Notebook(self)
        self.notebook.pack(fill="both", expand=True, padx=28, pady=16)
        self.tabs: dict[str, ChartTab] = {}
        chart_defs = [
            ("Performance", "portfolio_performance"),
            ("Correlation Heatmap", "correlation_heatmap"),
            ("Risk vs Return", "risk_return_scatter"),
            ("Allocation", "portfolio_allocation"),
        ]
        for label, name in chart_defs:
            tab = ChartTab(self.notebook, lambda n=name: self._make_chart(n), label)
            self.notebook.add(tab, text=label)
            self.tabs[name] = tab
        self.notebook.bind("<<NotebookTabChanged>>", self._on_tab_changed)
 
    def _make_chart(self, name: str) -> Figure:
        if self.price_frame is None:
            raise RuntimeError("No portfolio loaded")
        if name == "portfolio_allocation":
            n = len(self.price_frame.columns)
            weights = {t: round(100 / n, 2) for t in self.price_frame.columns}
            return ChartFactory.portfolio_allocation(weights)
        factory = getattr(ChartFactory, name)
        return factory(self.price_frame)
 
    def _on_tab_changed(self, _event):
        idx = self.notebook.index(self.notebook.select())
        name = list(self.tabs.keys())[idx]
        if self.price_frame is not None:
            self.tabs[name].render()
 
    def _compare(self):
        raw = [validate_ticker(t) for t in self.tickers_var.get().split(",")]
        tickers = [t for t in raw if t]
        if len(tickers) < 2:
            messagebox.showwarning("Input Needed", "Please enter at least two valid tickers.")
            return
        period = PERIOD_OPTIONS[self.period_var.get()]
        self.app.load_portfolio(tickers, period, on_done=self._on_loaded)
 
    def _on_loaded(self, price_frame: pd.DataFrame, metrics_frame: pd.DataFrame):
        self.price_frame = price_frame
        self.metrics_frame = metrics_frame
        self._on_tab_changed(None)
 
    def _export(self):
        if self.price_frame is None:
            messagebox.showinfo("Nothing to Export", "Run a comparison first.")
            return
        path = filedialog.asksaveasfilename(defaultextension=".xlsx",
                                             initialfile="portfolio_report.xlsx",
                                             filetypes=[("Excel Workbook", "*.xlsx")])
        if not path:
            return
        try:
            ExcelExporter.export_portfolio(path, self.price_frame, self.metrics_frame)
            messagebox.showinfo("Export Complete", f"Portfolio report saved to:\n{path}")
        except Exception as exc:
            messagebox.showerror("Export Failed", str(exc))
 
 
class ReportsPage(BasePage):
    def __init__(self, parent, app):
        super().__init__(parent, app)
        self._build()
 
    def _build(self):
        section_title(self, "Excel Reports").pack(anchor="w", padx=28, pady=(24, 4))
        tk.Label(self, text="Generate a fully formatted Excel workbook for the currently analyzed stock.",
                 font=Theme.FONT_SUBTITLE, bg=Theme.BG_PRIMARY, fg=Theme.TEXT_SECONDARY
                 ).pack(anchor="w", padx=28, pady=(0, 20))
 
        card = Card(self)
        card.pack(fill="x", padx=28, pady=6)
        self.info_var = tk.StringVar(value="No stock analyzed yet. Go to Dashboard and search a ticker first.")
        tk.Label(card, textvariable=self.info_var, font=Theme.FONT_BODY_BOLD, bg=Theme.BG_CARD,
                 fg=Theme.TEXT_PRIMARY, wraplength=600, justify="left").pack(anchor="w", padx=16, pady=16)
 
        feature_list = [
            "\u2713 Summary sheet with all key financial metrics",
            "\u2713 Full historical OHLCV data with computed daily returns",
            "\u2713 Conditional color-scale formatting on returns",
            "\u2713 Auto-sized columns for readability",
            "\u2713 Embedded native Excel line chart of closing price",
        ]
        for f in feature_list:
            tk.Label(card, text=f, font=Theme.FONT_BODY, bg=Theme.BG_CARD,
                     fg=Theme.TEXT_SECONDARY).pack(anchor="w", padx=16, pady=1)
 
        RoundedButton(card, "Generate Excel Report", command=self._export, width=220,
                      height=42).pack(anchor="w", padx=16, pady=18)
 
    def refresh(self, dataset: Optional[StockDataset]):
        if dataset:
            self.info_var.set(f"Ready to export report for {dataset.ticker}.")
        else:
            self.info_var.set("No stock analyzed yet. Go to Dashboard and search a ticker first.")
 
    def _export(self):
        dataset = self.app.current_dataset
        summary = self.app.current_summary
        if dataset is None or summary is None:
            messagebox.showinfo("No Data", "Please analyze a stock first (Dashboard tab).")
            return
        path = filedialog.asksaveasfilename(defaultextension=".xlsx",
                                             initialfile=f"{dataset.ticker}_report.xlsx",
                                             filetypes=[("Excel Workbook", "*.xlsx")])
        if not path:
            return
        try:
            ExcelExporter.export_single_stock(path, dataset, summary)
            self.app.storage.add_history({
                "ticker": dataset.ticker, "date": datetime.now().isoformat(timespec="seconds"),
                "type": "excel_export", "path": path,
            })
            messagebox.showinfo("Export Complete", f"Report saved to:\n{path}")
        except Exception as exc:
            messagebox.showerror("Export Failed", str(exc))
 
 
class WatchlistPage(BasePage):
    def __init__(self, parent, app):
        super().__init__(parent, app)
        self._build()
 
    def _build(self):
        section_title(self, "Watchlist & Analysis History").pack(anchor="w", padx=28, pady=(24, 4))
 
        body = tk.Frame(self, bg=Theme.BG_PRIMARY)
        body.pack(fill="both", expand=True, padx=28, pady=16)
 
        watch_card = Card(body)
        watch_card.pack(side="left", fill="both", expand=True, padx=(0, 12))
        tk.Label(watch_card, text="\u2605 Watchlist", font=Theme.FONT_H2, bg=Theme.BG_CARD,
                 fg=Theme.TEXT_PRIMARY).pack(anchor="w", padx=14, pady=(14, 6))
        self.watch_list = tk.Listbox(watch_card, bg=Theme.BG_INPUT, fg=Theme.ACCENT_AMBER,
                                      relief="flat", highlightthickness=0, font=Theme.FONT_BODY)
        self.watch_list.pack(fill="both", expand=True, padx=14, pady=(0, 6))
        btn_row = tk.Frame(watch_card, bg=Theme.BG_CARD)
        btn_row.pack(fill="x", padx=14, pady=(0, 14))
        RoundedButton(btn_row, "Open", command=self._open_selected, width=90, height=30,
                      font=Theme.FONT_SMALL).pack(side="left")
        RoundedButton(btn_row, "Remove", command=self._remove_selected, width=90, height=30,
                      bg=Theme.ACCENT_RED, hover_bg="#dc2626", font=Theme.FONT_SMALL).pack(side="left", padx=8)
 
        history_card = Card(body)
        history_card.pack(side="left", fill="both", expand=True, padx=(12, 0))
        tk.Label(history_card, text="\U0001F553 Analysis History", font=Theme.FONT_H2, bg=Theme.BG_CARD,
                 fg=Theme.TEXT_PRIMARY).pack(anchor="w", padx=14, pady=(14, 6))
        columns = ("ticker", "type", "date")
        self.history_tree = ttk.Treeview(history_card, columns=columns, show="headings", height=16)
        for col, w in zip(columns, (100, 130, 200)):
            self.history_tree.heading(col, text=col.capitalize())
            self.history_tree.column(col, width=w)
        self.history_tree.pack(fill="both", expand=True, padx=14, pady=(0, 14))
 
        self.refresh()
 
    def refresh(self):
        self.watch_list.delete(0, tk.END)
        for t in self.app.storage.data["watchlist"]:
            self.watch_list.insert(tk.END, t)
        self.history_tree.delete(*self.history_tree.get_children())
        for entry in self.app.storage.data["analysis_history"]:
            self.history_tree.insert("", "end", values=(entry.get("ticker"), entry.get("type"),
                                                          entry.get("date")))
 
    def _open_selected(self):
        sel = self.watch_list.curselection()
        if not sel:
            return
        ticker = self.watch_list.get(sel[0])
        self.app.dashboard_page.ticker_var.set(ticker)
        self.app.dashboard_page._search()
 
    def _remove_selected(self):
        sel = self.watch_list.curselection()
        if not sel:
            return
        ticker = self.watch_list.get(sel[0])
        self.app.storage.toggle_watchlist(ticker)
        self.refresh()
        self.app.dashboard_page.refresh_lists()
 
 
# =================================================================================
#  APPLICATION MODULE  (app.py)
# =================================================================================
class StockAnalyzerApp:
    """Main application controller: owns the window, pages, and background jobs."""
 
    def __init__(self, root: tk.Tk):
        self.root = root
        self.root.title("Stock Market Analyzer")
        self.root.geometry("1360x820")
        self.root.minsize(1080, 680)
        self.root.configure(bg=Theme.BG_PRIMARY)
 
        self.storage = AppStorage()
        self.fetcher = DataFetcher()
        self.task_queue: "queue.Queue" = queue.Queue()
 
        self.current_dataset: Optional[StockDataset] = None
        self.current_summary: Optional[dict] = None
 
        self._build_layout()
        self.root.after(120, self._poll_queue)
 
    # --- Layout -----------------------------------------------------------
    def _build_layout(self):
        container = tk.Frame(self.root, bg=Theme.BG_PRIMARY)
        container.pack(fill="both", expand=True)
 
        self.sidebar = Sidebar(container, on_navigate=self.navigate)
        self.sidebar.pack(side="left", fill="y")
 
        content_wrapper = tk.Frame(container, bg=Theme.BG_PRIMARY)
        content_wrapper.pack(side="left", fill="both", expand=True)
 
        self.content = tk.Frame(content_wrapper, bg=Theme.BG_PRIMARY)
        self.content.pack(fill="both", expand=True)
 
        self.status_bar = StatusBar(self.root)
        self.status_bar.pack(side="bottom", fill="x")
 
        self.dashboard_page = DashboardPage(self.content, self)
        self.analysis_page = AnalysisPage(self.content, self)
        self.portfolio_page = PortfolioPage(self.content, self)
        self.reports_page = ReportsPage(self.content, self)
        self.watchlist_page = WatchlistPage(self.content, self)
 
        self.pages = {
            "dashboard": self.dashboard_page,
            "analysis": self.analysis_page,
            "portfolio": self.portfolio_page,
            "reports": self.reports_page,
            "watchlist": self.watchlist_page,
        }
        self.navigate("dashboard")
 
        self.root.bind("<Control-e>", lambda e: self.reports_page._export())
 
    def navigate(self, key: str):
        for page in self.pages.values():
            page.pack_forget()
        page = self.pages[key]
        page.pack(fill="both", expand=True)
        if key == "reports":
            self.reports_page.refresh(self.current_dataset)
        if key == "watchlist":
            self.watchlist_page.refresh()
 
    # --- Background task orchestration -------------------------------------
    def _run_async(self, fn: Callable, on_success: Callable, status_msg: str):
        self.status_bar.start_progress(status_msg)
 
        def worker():
            try:
                result = fn()
                self.task_queue.put(("success", on_success, result))
            except Exception as exc:  # noqa: BLE001
                traceback.print_exc()
                self.task_queue.put(("error", None, str(exc)))
 
        threading.Thread(target=worker, daemon=True).start()
 
    def _poll_queue(self):
        try:
            while True:
                kind, callback, payload = self.task_queue.get_nowait()
                self.status_bar.stop_progress("Ready")
                if kind == "success":
                    callback(payload)
                else:
                    messagebox.showerror("Error", payload)
        except queue.Empty:
            pass
        self.root.after(150, self._poll_queue)
 
    # --- Public actions used by pages --------------------------------------
    def load_ticker(self, ticker: str, period: str, on_dashboard_update: Optional[Callable] = None):
        def fetch_and_compute():
            dataset = self.fetcher.fetch(ticker, period)
            summary = MetricsCalculator.build_summary(dataset)
            return dataset, summary
 
        def on_success(result):
            dataset, summary = result
            self.current_dataset = dataset
            self.current_summary = summary
            self.analysis_page.load(dataset, summary)
            self.storage.add_recent_search(ticker)
            self.storage.add_history({
                "ticker": ticker, "date": datetime.now().isoformat(timespec="seconds"),
                "type": "analysis",
            })
            if on_dashboard_update:
                on_dashboard_update(summary)
 
        self._run_async(fetch_and_compute, on_success, f"Fetching data for {ticker}...")
 
    def load_portfolio(self, tickers: list[str], period: str, on_done: Callable):
        def fetch_and_compute():
            price_data = {}
            metrics_rows = {}
            for ticker in tickers:
                dataset = self.fetcher.fetch(ticker, period, with_market=False)
                price_data[ticker] = dataset.history["Close"]
                returns = MetricsCalculator.daily_returns(dataset.history["Close"])
                metrics_rows[ticker] = {
                    "Annual Return %": returns.mean() * 252 * 100,
                    "Annual Volatility %": returns.std() * np.sqrt(252) * 100,
                    "Sharpe (rf=0)": (returns.mean() * 252) / (returns.std() * np.sqrt(252))
                        if returns.std() > 0 else np.nan,
                    "Max Drawdown %": ((1 + returns).cumprod() / (1 + returns).cumprod().cummax() - 1).min() * 100,
                }
            price_frame = pd.DataFrame(price_data).dropna(how="all")
            metrics_frame = pd.DataFrame(metrics_rows).T
            return price_frame, metrics_frame
 
        self._run_async(fetch_and_compute, lambda result: on_done(*result),
                         f"Comparing {', '.join(tickers)}...")
 
 
# =================================================================================
#  ENTRY POINT
# =================================================================================
def main():
    root = tk.Tk()
    root.withdraw()
 
    def launch_main_app():
        root.deiconify()
        StockAnalyzerApp(root)
 
    SplashScreen(root, on_finish=launch_main_app)
    root.mainloop()
 
 
if __name__ == "__main__":
    main()
